In [12]:
import json
import re
from pathlib import Path

In [13]:
# Muster: trifft auf digilib.bbaw.de/ nur wenn danach NICHT schon digilib/ steht
PATTERN = re.compile(r'(https?://digilib\.bbaw\.de/)(?!digilib/)')

def patch_node(node):
    if isinstance(node, dict):
        for key, value in node.items():
            if key in ('id', 'target') and isinstance(value, str):
                node[key] = PATTERN.sub(r'\1digilib/', value)
            else:
                patch_node(value)
    elif isinstance(node, list):
        for item in node:
            patch_node(item)
    return node

In [14]:
input_dir  = Path('../../data/manifest/raw')
output_dir = Path('../../data/manifest/curated')
output_dir.mkdir(exist_ok=True)

for input_path in sorted(input_dir.glob('**/*.json')):
    data = json.loads(input_path.read_text(encoding='utf-8'))
    patch_node(data)

    # Unterordner-Struktur im Ausgabe-Ordner beibehalten
    relative    = input_path.relative_to(input_dir)
    output_path = output_dir / relative
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

In [15]:
def migrate_iiif3_to_v2_service(data):
    """
    Durchsucht das JSON rekursiv nach 'ImageService3' Blöcken
    und konvertiert sie in das funktionierende IIIF v2 Format.
    """
    if isinstance(data, dict):
        # Falls es sich um einen v3 Service-Block handelt
        if data.get("type") == "ImageService3" and "id" in data:
            old_id = data["id"]
            # Entferne das /3/ aus der URL
            if "/IIIF/3/" in old_id:
                new_id = old_id.replace("/IIIF/3/", "/IIIF/")
                
                # Struktur auf v2 Syntax umbauen
                data.clear()
                data["@id"] = new_id
                data["@type"] = "iiif:ImageService2"
                data["profile"] = "http://iiif.io/api/image/2/level2.json"
                return
        
        # Falls ein 'service'-Array in einem Element liegt
        if "service" in data and isinstance(data["service"], list):
            for svc in data["service"]:
                if isinstance(svc, dict) and svc.get("type") == "ImageService3":
                    migrate_iiif3_to_v2_service(svc)
                    
        # Rekursiv weiter durch alle Schlüssel des Dicts gehen
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                migrate_iiif3_to_v2_service(value)
                
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, (dict, list)):
                migrate_iiif3_to_v2_service(item)

def process_manifest_folder(folder_path):
    """
    Liest alle .json/.jsonld Dateien im Ordner via pathlib, 
    konvertiert sie und ÜBERSCHREIBT die bestehenden Dateien.
    """
    folder = Path(folder_path)
    
    if not folder.exists():
        print(f"Fehler: Der Ordner '{folder}' existiert nicht.")
        return

    success_count = 0
    # iterdir() listet alle Dateien im Ordner auf
    for file_path in folder.rglob('*'):
        if file_path.is_file() and file_path.suffix in ['.json', '.jsonld']:
            try:
                # 1. Bestehende Datei einlesen
                with file_path.open("r", encoding="utf-8") as f:
                    manifest = json.load(f)
                
                # 2. Konvertierung im Speicher durchführen
                migrate_iiif3_to_v2_service(manifest)
                
                # 3. Exakt dieselbe Datei direkt ÜBERSCHREIBEN
                with file_path.open("w", encoding="utf-8") as f:
                    json.dump(manifest, f, indent=2, ensure_ascii=False)
                
                print(f"Erfolgreich überschrieben: {file_path.name}")
                success_count += 1
                
            except Exception as e:
                print(f"Fehler bei Datei {file_path.name}: {e}")
                
    print(f"\nFertig! {success_count} Manifest(e) im Ordner '{folder.resolve()}' wurden direkt überschrieben.")

# ==========================================
# EINSTELLUNG FÜR DEIN JUPYTER-NOTEBOOK
# ==========================================

ORDNER_PFAD = "../../data/manifest/curated"

# Skript starten
process_manifest_folder(ORDNER_PFAD)

Erfolgreich überschrieben: 05-mem_1796.json
Erfolgreich überschrieben: 05-mem_1798.json
Erfolgreich überschrieben: 05-mem_1804.json
Erfolgreich überschrieben: 05-mem_17881789.json
Erfolgreich überschrieben: 05-mem_17991800.json
Erfolgreich überschrieben: 05-mem_1801.json
Erfolgreich überschrieben: 05-mem_17901791.json
Erfolgreich überschrieben: 05-mem_17921793.json
Erfolgreich überschrieben: 05-mem_1802.json
Erfolgreich überschrieben: 05-mem_17941795.json
Erfolgreich überschrieben: 05-mem_17861787.json
Erfolgreich überschrieben: 05-mem_1797.json
Erfolgreich überschrieben: 05-mem_1803.json
Erfolgreich überschrieben: 01-misc_3.json
Erfolgreich überschrieben: 01-misc_1.json
Erfolgreich überschrieben: 01-misc_5.json
Erfolgreich überschrieben: 01-misc_6.json
Erfolgreich überschrieben: 01-misc_7.json
Erfolgreich überschrieben: 01-misc_2.json
Erfolgreich überschrieben: 01-misc_4.json
Erfolgreich überschrieben: 01-misc_7-s.json
Erfolgreich überschrieben: 10-sitz_1889-2.json
Erfolgreich übersch